# Credit Risk Modeling — Annotated Walkthrough

**Goal of this notebook:** predict which loan customers are likely to be *"bad"* (high credit risk) so the lender can decide whether to **Approve (0)** or **Deny (1)** an application.

**The overall flow:**
1. Load a 10,000-customer sample.
2. Define the target variable `bad` and build the feature matrix `X`.
3. Train and evaluate a **Decision Tree**, being careful about *data leakage* and *derogatory-history features*.
4. Cross-validate, tune hyperparameters, and pick a decision threshold.
5. Interpret the model (feature importances, permutation importance, SHAP).
6. Compare against a **Random Forest** and visualize confusion matrices.

*The descriptions below were added to explain each step; the code itself is unchanged.*

## 1. Load the data (and the sampling story behind it)

Reads a **10,000-row sample** of customer data from an Excel workbook (the `'Customer Data'` sheet) into a DataFrame `df`, then prints its shape.

The long comment block documents *how* this sample was drawn: a simple 10% random sample of the full 100k dataset, seeded with `random_state=42` for reproducibility. Because "bad" outcomes are rare (~1.6%), the author notes that simple random sampling doesn't *guarantee* the rare-class rate is preserved — they sanity-checked it landed at 1.59%.

In [ ]:
# --- Sampling Methodology ---
# The full dataset contains 100,000 customer rows. For faster iteration during
# model development, we used SIMPLE RANDOM SAMPLING to draw a 10,000-row subset
# (10% of the full data), with no stratification by outcome or any attribute.
#
# random_state=42 fixes the seed so this exact same sample is reproducible
# across team members and re-runs.
#
# Note: because the "bad" outcome rate is only ~1.6% in the full dataset, simple
# random sampling does not guarantee the sample preserves that exact rate
# (stratified sampling would be needed for that guarantee). We verified the
# sample's bad rate (1.59%) closely matched the full dataset's rate as a sanity check.
import pandas as pd

df = pd.read_excel('/content/sample_data/credit_risk_dataset_v2_sample10k.xlsx', sheet_name='Customer Data')
print(df.shape)

## 2. Define the target `bad` and build the feature matrix — while avoiding leakage

**Creates the label:** a customer is flagged `bad = 1` if *any* of three risk attributes cross a threshold (one ≥ 8, two others > 0), else `0`. `value_counts()` and `mean()` show the class balance and the resulting bad rate.

**Prevents data leakage:** the three columns used to *build* the label are stored in `leakage_cols` and dropped from the features — otherwise the model could "cheat" by learning the exact rule that defined the target. `customer_id` (an identifier, not a predictor) is also dropped.

The result is `X` (features) and `y` (target). The final line confirms there are no missing values.

In [ ]:

df['bad'] = (
    (df['risk_attribute_383060'] >= 8) |
    (df['risk_attribute_274389'] > 0) |
    (df['risk_attribute_272634'] > 0)
).astype(int)
print(df['bad'].value_counts())
print("Bad rate:", df['bad'].mean())

leakage_cols = ['risk_attribute_383060', 'risk_attribute_274389', 'risk_attribute_272634']
X = df.drop(columns=['customer_id', 'bad'] + leakage_cols)
y = df['bad']
print("X shape:", X.shape, "y shape:", y.shape)
print(X.isnull().sum().sum())

## 3. Train/test split

Splits the data 80% train / 20% test. `stratify=y` keeps the same bad rate in both halves (important because the classes are imbalanced), and `random_state=42` makes the split reproducible. The prints confirm the sizes and that the bad rate matches across train and test.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(X_train.shape, X_test.shape)
print(y_train.mean(), y_test.mean())

## 4. Train a first Decision Tree

Fits a `DecisionTreeClassifier`:
- `max_depth=5` keeps the tree shallow so it's interpretable and less prone to overfitting.
- `class_weight='balanced'` up-weights the rare "bad" class so the model doesn't just predict "Approve" for everyone.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

tree = DecisionTreeClassifier(max_depth=5, class_weight='balanced', random_state=42)
tree.fit(X_train, y_train)

## 5. Evaluate the first tree

Scores the model on the held-out test set:
- **ROC-AUC** — overall ranking ability (how well predicted risk separates good vs. bad).
- **classification report** — precision/recall/F1 for each class.
- **confusion matrix** — raw counts of correct vs. incorrect Approve/Deny decisions.

`predict_proba(...)[:, 1]` grabs the predicted *probability* of the "bad" class, used for ROC-AUC.

In [ ]:
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix

y_pred = tree.predict(X_test)
y_proba = tree.predict_proba(X_test)[:, 1]

print("ROC-AUC:", roc_auc_score(y_test, y_proba))
print(classification_report(y_test, y_pred, target_names=['Approve (0)', 'Deny (1)']))
print(confusion_matrix(y_test, y_pred))

## 6. Which features did the tree rely on?

Prints the tree's built-in **feature importances** (non-zero only, sorted). This shows which attributes drove the splits — a quick first look at what the model considers predictive.

In [ ]:
importances = pd.Series(tree.feature_importances_, index=X.columns)
print(importances[importances > 0].sort_values(ascending=False))

## 7. Raw correlations with the target

Separately from the model, computes the absolute **correlation of each feature with `y`** and shows the top 20. A simple, model-independent check of which variables move together with the bad outcome.

In [ ]:
correlations = X.corrwith(y).abs().sort_values(ascending=False)
print(correlations.head(20))

## 8. Exclude "derogatory history" features

This is a **fairness / policy step.** It reads the workbook's `'Data Dictionary'` sheet and flags any column that describes *past derogatory credit history* — bankruptcies, charge-offs, collections, delinquencies, liens, etc. — using both a set of dictionary **categories** and a list of **keywords** matched against each column's definition.

Those columns are collected into `exclude_cols` and dropped to create `X_clean`. The intent is to build a model that predicts risk **without leaning on prior derogatory-history flags** (which can bake in past adverse events). The prints report how many columns were excluded and the new feature-matrix shape.

In [ ]:
dd = pd.read_excel('/content/sample_data/credit_risk_dataset_v2_sample10k.xlsx', sheet_name='Data Dictionary')

derog_categories = {
    'Worst Status Codes', 'Collections and Derogatory Metrics', 'Public Records and Bankruptcy',
    'Financial Stress and Hardship Indicators', 'Months Since Delinquency',
    'Delinquency Counts – 30 Days', 'Delinquency Counts – 60 Days and Severe',
    'Delinquency Percentages', 'Additional Delinquency Severity and Recurrence'
}
keywords = ['charge-off','chargeoff','bankrupt','delinquen','derogatory','collection',
            'default','write-off','writeoff','garnishment','hardship','nsf','judgment',
            'lien','forbearance','deferment','settled']

def is_derogatory(row):
    cat = row['Category']
    definition = str(row['Definition']).lower()
    return cat in derog_categories or any(k in definition for k in keywords)

exclude_cols = dd[dd.apply(is_derogatory, axis=1)]['Column Name'].tolist()
print(f"Excluding {len(exclude_cols)} derogatory-history columns")

X_clean = df.drop(columns=['customer_id', 'bad'] + exclude_cols, errors='ignore')
y = df['bad']
print("X_clean shape:", X_clean.shape)

## 9. Re-split, re-train, and re-evaluate on the cleaned features

Same pipeline as before — split → fit `DecisionTreeClassifier` → score — but now on `X_clean` (derogatory-history features removed). The model is named `tree_clean`. Comparing its ROC-AUC to the earlier tree shows how much predictive power those excluded features were providing.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix

X_train, X_test, y_train, y_test = train_test_split(
    X_clean, y, test_size=0.2, random_state=42, stratify=y
)

tree_clean = DecisionTreeClassifier(max_depth=5, class_weight='balanced', random_state=42)
tree_clean.fit(X_train, y_train)

y_pred = tree_clean.predict(X_test)
y_proba = tree_clean.predict_proba(X_test)[:, 1]

print("ROC-AUC:", roc_auc_score(y_test, y_proba))
print(classification_report(y_test, y_pred, target_names=['Approve (0)', 'Deny (1)']))
print(confusion_matrix(y_test, y_pred))

## 10. Feature importances for the cleaned tree

Same as step 6, but for `tree_clean`: which of the *remaining* features the model now relies on.

In [ ]:
importances = pd.Series(tree_clean.feature_importances_, index=X_clean.columns)
print(importances[importances > 0].sort_values(ascending=False))

## 11. Visualize the decision tree

Draws the full `tree_clean` with `plot_tree` (colored, rounded nodes), saves it as `decision_tree.png`, and displays it. This makes the model's actual decision rules human-readable — you can trace the path from root to leaf.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.tree import plot_tree

plt.figure(figsize=(28, 14))
plot_tree(tree_clean, feature_names=X_clean.columns, class_names=['Approve','Deny'],
          filled=True, rounded=True, fontsize=8)
plt.savefig('decision_tree.png', dpi=150, bbox_inches='tight')
plt.show()


## 12. Cross-validation

Runs **5-fold stratified cross-validation** (ROC-AUC) so the performance estimate doesn't depend on one lucky train/test split. Prints the five fold scores plus their mean and standard deviation — the std tells you how stable the model is.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(tree_clean, X_clean, y, cv=cv, scoring='roc_auc')

print("CV ROC-AUC scores:", cv_scores)
print(f"Mean: {cv_scores.mean():.4f}  Std: {cv_scores.std():.4f}")

## 13. Hyperparameter tuning with grid search

Uses `GridSearchCV` to try every combination of `max_depth`, `min_samples_leaf`, and `min_samples_split`, scoring each with 5-fold ROC-AUC. Reports the best-performing settings and stores the winning model as `best_tree`.

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier

param_grid = {
    'max_depth': [3, 4, 5, 6, 8, 10],
    'min_samples_leaf': [5, 10, 20, 50],
    'min_samples_split': [10, 20, 50]
}

grid = GridSearchCV(
    DecisionTreeClassifier(class_weight='balanced', random_state=42),
    param_grid,
    scoring='roc_auc',
    cv=5,
    n_jobs=-1
)
grid.fit(X_clean, y)

print("Best params:", grid.best_params_)
print("Best CV ROC-AUC:", grid.best_score_)

best_tree = grid.best_estimator_

## 14. Precision vs. recall across thresholds

By default a classifier denies when predicted probability ≥ 0.5, but that cutoff is a *choice*. This plots how **precision** and **recall** trade off as the decision threshold changes — helping pick a cutoff that matches business priorities (e.g., catching more bad loans vs. avoiding wrongly denying good customers).

In [ ]:
from sklearn.metrics import precision_recall_curve

precision, recall, thresholds = precision_recall_curve(y_test, y_proba)

plt.figure(figsize=(8,5))
plt.plot(thresholds, precision[:-1], label='Precision')
plt.plot(thresholds, recall[:-1], label='Recall')
plt.xlabel('Decision threshold (probability cutoff for Deny)')
plt.ylabel('Score')
plt.legend()
plt.title('Precision vs. Recall across decision thresholds')
plt.show()

## 15. Apply a custom decision threshold

Sets the cutoff to **0.7** (deny only when the model is fairly confident), recomputes predictions, and prints the classification report and confusion matrix at that threshold. You can change `chosen_threshold` and re-run to compare.

In [ ]:
chosen_threshold = 0.7  # try adjusting this and re-running to compare

y_pred_custom = (y_proba >= chosen_threshold).astype(int)

print(f"At threshold = {chosen_threshold}:")
print(classification_report(y_test, y_pred_custom, target_names=['Approve (0)', 'Deny (1)']))
print(confusion_matrix(y_test, y_pred_custom))

## 16. Calibration check

A **calibration curve** asks: when the model says "30% chance of default," do about 30% of those customers actually default? Points near the diagonal mean the predicted probabilities are trustworthy as *actual* risk estimates, not just rankings.

In [ ]:
from sklearn.calibration import calibration_curve

prob_true, prob_pred = calibration_curve(y_test, y_proba, n_bins=10, strategy='quantile')

plt.figure(figsize=(6,6))
plt.plot(prob_pred, prob_true, marker='o', label='Decision Tree')
plt.plot([0,1], [0,1], linestyle='--', color='gray', label='Perfectly calibrated')
plt.xlabel('Predicted probability of default')
plt.ylabel('Actual observed default rate')
plt.legend()
plt.title('Calibration: predicted risk vs. actual outcomes')
plt.show()

## 17. Permutation importance

A more reliable importance measure than the tree's built-in one: it shuffles each feature and measures how much ROC-AUC *drops*. Features that hurt performance most when scrambled are the ones the model genuinely depends on. Shows the top 15.

In [ ]:
from sklearn.inspection import permutation_importance

result = permutation_importance(tree_clean, X_test, y_test, n_repeats=10, random_state=42, scoring='roc_auc')

perm_importances = pd.Series(result.importances_mean, index=X_clean.columns).sort_values(ascending=False)
print(perm_importances.head(15))

## 18. SHAP explanations

Installs and uses **SHAP** to explain predictions. `TreeExplainer` computes each feature's contribution to individual predictions, and `summary_plot` shows the top 15 features by overall impact plus the *direction* of their effect (which values push toward Approve vs. Deny).

In [ ]:
!pip install shap -q
import shap

explainer = shap.TreeExplainer(tree_clean)
shap_values = explainer(X_test)

shap.summary_plot(shap_values[:,:,1], X_test, max_display=15)


## 19. (Repeat) Apply custom threshold

⚠️ *This cell is a duplicate of step 15 — same threshold logic.*

In [ ]:
chosen_threshold = 0.7  # try adjusting this and re-running to compare

y_pred_custom = (y_proba >= chosen_threshold).astype(int)

print(f"At threshold = {chosen_threshold}:")
print(classification_report(y_test, y_pred_custom, target_names=['Approve (0)', 'Deny (1)']))
print(confusion_matrix(y_test, y_pred_custom))

## 20. (Repeat) Calibration check

⚠️ *This cell is a duplicate of step 16.*

In [ ]:
from sklearn.calibration import calibration_curve

prob_true, prob_pred = calibration_curve(y_test, y_proba, n_bins=10, strategy='quantile')

plt.figure(figsize=(6,6))
plt.plot(prob_pred, prob_true, marker='o', label='Decision Tree')
plt.plot([0,1], [0,1], linestyle='--', color='gray', label='Perfectly calibrated')
plt.xlabel('Predicted probability of default')
plt.ylabel('Actual observed default rate')
plt.legend()
plt.title('Calibration: predicted risk vs. actual outcomes')
plt.show()

## 21. (Repeat) Permutation importance

⚠️ *This cell is a duplicate of step 17.*

In [ ]:
from sklearn.inspection import permutation_importance

result = permutation_importance(tree_clean, X_test, y_test, n_repeats=10, random_state=42, scoring='roc_auc')

perm_importances = pd.Series(result.importances_mean, index=X_clean.columns).sort_values(ascending=False)
print(perm_importances.head(15))


## 22. (Repeat) SHAP explanations

⚠️ *This cell is a duplicate of step 18.*

In [ ]:
!pip install shap -q
import shap

explainer = shap.TreeExplainer(tree_clean)
shap_values = explainer(X_test)

shap.summary_plot(shap_values[:,:,1], X_test, max_display=15)

## 23. Random Forest model

Trains a **Random Forest** (200 trees, `max_depth=8`, balanced classes) — an ensemble that averages many trees and usually outperforms a single tree. Fits on the current `X_train`/`y_train` and prints ROC-AUC, classification report, and confusion matrix.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
y_proba_rf = rf.predict_proba(X_test)[:, 1]

print("Random Forest ROC-AUC:", roc_auc_score(y_test, y_proba_rf))
print(classification_report(y_test, y_pred_rf, target_names=['Approve (0)', 'Deny (1)']))
print(confusion_matrix(y_test, y_pred_rf))


## 24. Compare Decision Tree vs. Random Forest

Builds a small results table putting the single tree's ROC-AUC next to the random forest's, for a side-by-side comparison.

In [ ]:
results = pd.DataFrame([
    {'Model': 'Decision Tree', 'ROC-AUC': roc_auc_score(y_test, y_proba)},
    {'Model': 'Random Forest', 'ROC-AUC': roc_auc_score(y_test, y_proba_rf)},
])
print(results)

## 25. (Repeat) Random Forest model

⚠️ *This cell is a duplicate of step 23.*

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
y_proba_rf = rf.predict_proba(X_test)[:, 1]

print("Random Forest ROC-AUC:", roc_auc_score(y_test, y_proba_rf))
print(classification_report(y_test, y_pred_rf, target_names=['Approve (0)', 'Deny (1)']))
print(confusion_matrix(y_test, y_pred_rf))

## 26. (Repeat) Model comparison table

⚠️ *This cell is a duplicate of step 24.*

In [ ]:
results = pd.DataFrame([
    {'Model': 'Decision Tree', 'ROC-AUC': roc_auc_score(y_test, y_proba)},
    {'Model': 'Random Forest', 'ROC-AUC': roc_auc_score(y_test, y_proba_rf)},
])
print(results)

## 27. Confusion-matrix plotting helper

Defines a reusable `plot_confusion(...)` function that draws a labeled Approve/Deny confusion matrix, then calls it on an "optimized tree with feature selection."

⚠️ **Heads up:** this line references `best_tree_selected` and `X_test_selected`, which are **never defined anywhere in this notebook** — so this cell will raise a `NameError` as-is. It looks like a feature-selection step was planned or removed. You'd need to define those (or swap in `best_tree` / `X_test`) for it to run.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
import matplotlib.pyplot as plt

def plot_confusion(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Approve', 'Deny'])
    disp.plot(cmap='Blues', values_format='d')
    plt.title(title)
    plt.show()

# Optimized tree with feature selection
plot_confusion(y_test, best_tree_selected.predict(X_test_selected), 'Optimized Decision Tree (20 features)')

## 28. Confusion matrices side by side

Uses the helper above to plot confusion matrices for three models: the cleaned decision tree, the random forest, and the grid-search `best_tree`. A visual way to compare *what kind* of mistakes each model makes (missed bad loans vs. wrongly denied good ones).

In [ ]:
plot_confusion(y_test, tree_clean.predict(X_test), 'Decision Tree (all 244 features)')
plot_confusion(y_test, rf.predict(X_test), 'Random Forest')
plot_confusion(y_test, best_tree.predict(X_test), 'Optimized Decision Tree (all features)')
